# 17: Developer Tooling

This notebook demonstrates the **meta-tooling** modules of siege_utilities --
the utilities that support library maintenance, code quality, and release workflows
rather than data processing.

## Modules Covered

| Module | Purpose |
|--------|---------|
| `development/architecture.py` | Package structure analysis and architecture diagrams |
| `git/branch_analyzer.py` | Branch status, commit history, file change analysis |
| `git/git_status.py` | Repository status, branch info, remote info, repo size |
| `git/git_workflow.py` | Branch naming validation, commit message conventions |
| `testing/runner.py` | Smoke tests, test report generation |
| `testing/environment.py` | System info, environment diagnostics |
| `hygiene/generate_docstrings.py` | Function signature analysis, docstring templates |
| `hygiene/pypi_release.py` | Version management, release readiness checks |

## Safety

All git operations in this notebook are **read-only**. No branches are created,
no commits are made, and no pushes are executed. The release section uses
`dry_run=True` and `check_release_readiness()` only -- nothing is uploaded.

## Prerequisites

```bash
pip install -e /path/to/siege_utilities  # Standard install is sufficient
```

## 1. Setup & Imports

In [ ]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')
sys.path.insert(0, str(Path.cwd().parent))

import siege_utilities as su
su.configure_shared_logging(level="INFO")

su.log_info("Base imports successful")

# --- Path configuration ---
from siege_utilities.config.user_config import get_user_config

DATA_DIR = Path(get_user_config().get_download_directory())
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Repo path for git operations (read-only)
REPO_PATH = str(Path.cwd().parent)

su.log_info(f"Data directory: {DATA_DIR}  (exists={DATA_DIR.exists()})")
su.log_info(f"Repo path:     {REPO_PATH}")

## 2. Package Architecture Analysis

The `development/architecture.py` module introspects the siege_utilities package at
runtime to produce a structural overview: modules, functions, classes, and their
relationships. Output formats include plain text, JSON, and Markdown.

In [ ]:
from siege_utilities.development.architecture import (
    analyze_package_structure,
    generate_architecture_diagram,
)

# --- 2a. Analyze package structure ---
structure = analyze_package_structure("siege_utilities")

su.log_info("Package Structure Analysis")
su.log_info("=" * 50)
su.log_info(f"Package:          {structure.get('package_name', 'N/A')}")
su.log_info(f"Package path:     {structure.get('package_path', 'N/A')}")
su.log_info(f"Total modules:    {structure.get('module_count', 0)}")
su.log_info(f"Total functions:  {structure.get('total_functions', 0)}")
su.log_info(f"Total classes:    {structure.get('total_classes', 0)}")

# List discovered modules
su.log_info("")
su.log_info("Discovered modules:")
for mod_name, mod_info in structure.get('modules', {}).items():
    func_count = mod_info.get('function_count', 0)
    class_count = mod_info.get('class_count', 0)
    su.log_info(f"  {mod_name:30s}  functions={func_count:3d}  classes={class_count:3d}")

In [ ]:
# --- 2b. Generate architecture diagram (text format) ---
text_diagram = generate_architecture_diagram(output_format="text", include_details=True)
print(text_diagram)

In [ ]:
# --- 2c. Generate architecture diagram (markdown format) ---
md_diagram = generate_architecture_diagram(output_format="markdown", include_details=False)
print(md_diagram[:2000])  # Truncate for display

## 3. Git Repository Analysis

The `git/` subpackage provides read-only repository introspection. All functions
accept a `repo_path` argument; we point them at the siege_utilities repo root.

**All operations here are strictly read-only.**

In [ ]:
from siege_utilities.git.branch_analyzer import (
    analyze_branch_status,
    get_commit_history,
    get_file_changes,
    generate_branch_report,
)
from siege_utilities.git.git_status import (
    get_repository_status,
    get_branch_info,
    get_remote_info,
    get_file_status,
    get_repository_size,
)

su.log_info("Git analysis imports successful")

In [ ]:
# --- 3a. Branch status ---
branch_status = analyze_branch_status(repo_path=REPO_PATH)

su.log_info("Branch Status")
su.log_info("=" * 50)
for key, value in branch_status.items():
    su.log_info(f"  {key:25s}: {value}")

In [ ]:
# --- 3b. Recent commit history ---
commits = get_commit_history(limit=10, repo_path=REPO_PATH)

su.log_info(f"Recent Commits ({len(commits)} shown)")
su.log_info("=" * 80)
for c in commits:
    su.log_info(f"  {c['hash']}  {c['date']}  {c['author']:20s}  {c['message'][:60]}")

In [ ]:
# --- 3c. File changes from main ---
file_changes = get_file_changes(repo_path=REPO_PATH)

su.log_info("File Changes (vs main)")
su.log_info("=" * 50)
su.log_info(f"  Added:    {len(file_changes.get('added', []))} files")
su.log_info(f"  Modified: {len(file_changes.get('modified', []))} files")
su.log_info(f"  Deleted:  {len(file_changes.get('deleted', []))} files")

# Show first few of each
for status in ['added', 'modified', 'deleted']:
    files = file_changes.get(status, [])
    if files:
        su.log_info(f"\n  {status.title()}:")
        for f in files[:5]:
            su.log_info(f"    {f}")
        if len(files) > 5:
            su.log_info(f"    ... and {len(files) - 5} more")

In [ ]:
# --- 3d. Repository status (comprehensive) ---
repo_status = get_repository_status(repo_path=REPO_PATH)

su.log_info("Repository Status")
su.log_info("=" * 50)
su.log_info(f"  Branch:            {repo_status.get('current_branch', 'N/A')}")
su.log_info(f"  Clean:             {repo_status.get('working_directory_clean', 'N/A')}")
su.log_info(f"  Staged files:      {repo_status.get('staged_files', 0)}")
su.log_info(f"  Unstaged files:    {repo_status.get('unstaged_files', 0)}")
su.log_info(f"  Untracked files:   {repo_status.get('untracked_files', 0)}")

last_commit = repo_status.get('last_commit', {})
su.log_info(f"  Last commit:       {last_commit.get('hash', 'N/A')} ({last_commit.get('date', 'N/A')})")
su.log_info(f"  Commit message:    {last_commit.get('message', 'N/A')[:60]}")

remote = repo_status.get('remote', {})
su.log_info(f"  Remote:            {remote.get('url', 'N/A')}")
su.log_info(f"  Ahead/Behind:      +{remote.get('ahead', 0)} / -{remote.get('behind', 0)}")

In [ ]:
# --- 3e. Branch info ---
branch_info = get_branch_info(repo_path=REPO_PATH)

su.log_info("Branch Info")
su.log_info("=" * 50)
su.log_info(f"  Current branch:       {branch_info.get('current_branch', 'N/A')}")
su.log_info(f"  Local branches:       {branch_info.get('total_local_branches', 0)}")
su.log_info(f"  Remote branches:      {branch_info.get('total_remote_branches', 0)}")

su.log_info("")
su.log_info("  Local branches:")
for branch in branch_info.get('local_branches', []):
    name = branch.get('name', 'N/A')
    upstream = branch.get('upstream', 'none')
    lc = branch.get('last_commit', {})
    lc_hash = lc.get('hash', 'N/A') if lc else 'N/A'
    su.log_info(f"    {name:40s}  upstream={upstream or 'none':30s}  last={lc_hash}")

In [ ]:
# --- 3f. Remote info ---
remote_info = get_remote_info(repo_path=REPO_PATH)

su.log_info("Remote Info")
su.log_info("=" * 50)
su.log_info(f"  Total remotes:     {remote_info.get('total_remotes', 0)}")
su.log_info(f"  Default remote:    {remote_info.get('default_remote', 'N/A')}")

for remote in remote_info.get('remotes', []):
    su.log_info(f"  {remote.get('name', 'N/A'):15s}  {remote.get('url', 'N/A')}")

In [ ]:
# --- 3g. Repository size ---
repo_size = get_repository_size(repo_path=REPO_PATH)

su.log_info("Repository Size")
su.log_info("=" * 50)
su.log_info(f"  .git directory:    {repo_size.get('git_directory_size_mb', 0):.2f} MB")
su.log_info(f"  Working tree:      {repo_size.get('working_directory_size_mb', 0):.2f} MB")
su.log_info(f"  Total:             {repo_size.get('total_size_mb', 0):.2f} MB")

In [ ]:
# --- 3h. Full branch report ---
report = generate_branch_report(repo_path=REPO_PATH)

# Show the first 80 lines of the report
report_lines = report.split('\n')
su.log_info(f"Branch report generated ({len(report_lines)} lines total)")
su.log_info("Showing first 40 lines:")
print('\n'.join(report_lines[:40]))

## 4. Git Workflow Validation

The `git/git_workflow.py` module provides validators for branch naming conventions
and commit message formats. These are pure string-validation functions -- they
do not perform any actual git operations.

**No git write operations are performed here.**

In [ ]:
from siege_utilities.git.git_workflow import (
    validate_branch_naming,
    enforce_commit_conventions,
)

# --- 4a. Branch naming validation ---
test_branches = [
    "feature/add-voter-file-parser",
    "bugfix/fix-null-handling",
    "hotfix/critical-db-fix",
    "release/1.2.3",
    "chore/update-dependencies",
    # Invalid examples
    "my-random-branch",
    "Feature/CamelCase-Bad",
    "feature/this-branch-name-is-way-too-long-and-exceeds-the-fifty-character-maximum-limit",
    "-starts-with-hyphen",
]

su.log_info("Branch Naming Validation")
su.log_info("=" * 70)
for branch in test_branches:
    result = validate_branch_naming(branch)
    status = "VALID" if result['is_valid'] else "INVALID"
    pattern = result.get('matched_pattern', 'none')
    su.log_info(f"  [{status:7s}] {branch:55s}  pattern={pattern}")
    if result.get('issues'):
        for issue in result['issues']:
            su.log_info(f"           Issue: {issue}")
    if result.get('suggestions'):
        for suggestion in result['suggestions'][:2]:
            su.log_info(f"           Suggestion: {suggestion}")

In [ ]:
# --- 4b. Commit message validation ---
test_messages = [
    "feat(parser): add FEC schedule B support",
    "fix: resolve null pointer in voter merge",
    "docs: update API reference for v2",
    "refactor(geo): simplify boundary lookup",
    "test(runner): add smoke test for git module",
    # Invalid examples
    "Updated the code",
    "WIP: still working on this",
    "Fixed stuff",
    "This commit message is way too long and should be shortened because the first line exceeds the seventy-two character limit",
]

su.log_info("Commit Message Validation")
su.log_info("=" * 70)
for msg in test_messages:
    result = enforce_commit_conventions(msg)
    status = "VALID" if result['is_valid'] else "INVALID"
    conventional = "conventional" if result.get('is_conventional') else "non-conventional"
    su.log_info(f"  [{status:7s}] {msg[:60]:60s}  ({conventional})")
    if result.get('issues'):
        for issue in result['issues']:
            su.log_info(f"           Issue: {issue}")
    if result.get('suggestions'):
        for suggestion in result['suggestions'][:2]:
            su.log_info(f"           Suggestion: {suggestion}")

## 5. Test Environment & Runner

The `testing/` subpackage provides environment diagnostics and test orchestration.
`get_system_info()` collects platform details useful for debugging, while
`diagnose_environment()` validates that required tooling (Java, Spark, etc.) is
available. `quick_smoke_test()` verifies basic package functionality.

In [ ]:
from siege_utilities.testing.environment import (
    get_system_info,
    diagnose_environment,
)
from siege_utilities.testing.runner import (
    quick_smoke_test,
    get_test_report,
)

su.log_info("Testing module imports successful")

In [ ]:
# --- 5a. System info ---
sys_info = get_system_info()

su.log_info("System Information")
su.log_info("=" * 60)
for key, value in sys_info.items():
    # Truncate long values like python_version
    display_value = str(value)[:80]
    su.log_info(f"  {key:30s}: {display_value}")

In [ ]:
# --- 5b. Environment diagnostics ---
su.log_info("Running environment diagnostics...")
env_healthy = diagnose_environment()
su.log_info(f"\nEnvironment healthy: {env_healthy}")

In [ ]:
# --- 5c. Quick smoke test ---
su.log_info("Running quick smoke test...")
smoke_passed = quick_smoke_test()
su.log_info(f"\nSmoke test passed: {smoke_passed}")

In [ ]:
# --- 5d. Full test report ---
report = get_test_report()

su.log_info("Test Report")
su.log_info("=" * 50)
su.log_info(f"  Environment healthy:  {report.get('environment_healthy', 'N/A')}")
su.log_info(f"  Smoke test passed:    {report.get('smoke_test_passed', 'N/A')}")

deps = report.get('dependencies', {})
if deps:
    available = sum(1 for v in deps.values() if v)
    su.log_info(f"  Dependencies:         {available}/{len(deps)} available")

suggestions = report.get('suggestions', [])
if suggestions:
    su.log_info("\n  Suggestions:")
    for s in suggestions:
        su.log_info(f"    - {s}")
else:
    su.log_info("  No suggestions -- everything looks good")

## 6. Docstring Hygiene

The `hygiene/generate_docstrings.py` module provides tools for analyzing function
signatures and generating standardized docstring templates. This is useful for
maintaining consistent documentation across the codebase.

These functions inspect live Python objects -- they do not modify any files.

In [ ]:
from siege_utilities.hygiene.generate_docstrings import (
    analyze_function_signature,
    generate_docstring_template,
)

su.log_info("Docstring hygiene imports successful")

In [ ]:
# --- 6a. Analyze function signatures ---
# Use some real functions from the modules we just imported
from siege_utilities.git.git_status import get_repository_status, get_repository_size
from siege_utilities.development.architecture import analyze_package_structure

sample_functions = [
    ("get_repository_status", get_repository_status),
    ("get_repository_size", get_repository_size),
    ("analyze_package_structure", analyze_package_structure),
]

su.log_info("Function Signature Analysis")
su.log_info("=" * 60)
for func_name, func_obj in sample_functions:
    params, return_desc = analyze_function_signature(func_obj)
    su.log_info(f"\n  {func_name}:")
    su.log_info(f"    Parameters:")
    for p in params:
        su.log_info(f"      - {p}")
    su.log_info(f"    Returns: {return_desc}")

In [ ]:
# --- 6b. Generate docstring templates ---
su.log_info("Generated Docstring Templates")
su.log_info("=" * 60)

for func_name, func_obj in sample_functions:
    template = generate_docstring_template(func_name, func_obj)
    su.log_info(f"\n--- Template for {func_name} ---")
    print(template)
    print()

In [ ]:
# --- 6c. Generate template without a live object (name-based heuristics only) ---
name_only_functions = [
    "download_remote_file",
    "validate_geojson_geometry",
    "configure_spark_session",
    "clean_string_column",
]

su.log_info("Name-based Docstring Templates (no live object)")
su.log_info("=" * 60)
for func_name in name_only_functions:
    template = generate_docstring_template(func_name)
    su.log_info(f"\n--- {func_name} ---")
    # Show just the first few lines
    lines = template.split('\n')
    for line in lines[:5]:
        su.log_info(f"  {line}")
    if len(lines) > 5:
        su.log_info(f"  ... ({len(lines) - 5} more lines)")

## 7. Release Management

The `hygiene/pypi_release.py` module manages package versioning and release
workflows. Here we use only the **read-only / dry-run** functions:

- `get_current_version()` -- reads version from setup.py
- `check_release_readiness()` -- validates that required files exist

**No actual uploads are performed. `full_release_workflow` is shown with `dry_run=True` only.**

In [ ]:
from siege_utilities.hygiene.pypi_release import (
    get_current_version,
    check_release_readiness,
)

su.log_info("Release management imports successful")

In [ ]:
# --- 7a. Get current version ---
import os
setup_py_path = os.path.join(REPO_PATH, "setup.py")

try:
    current_version = get_current_version(setup_py_path)
    su.log_info(f"Current package version: {current_version}")
except FileNotFoundError:
    su.log_warning(f"setup.py not found at {setup_py_path}")
    current_version = None
except ValueError as e:
    su.log_warning(f"Could not extract version: {e}")
    current_version = None

In [ ]:
# --- 7b. Check release readiness ---
# Note: This checks for files in the CWD, so we show what it finds
# from the notebook directory context
original_cwd = os.getcwd()
os.chdir(REPO_PATH)

try:
    is_ready, issues = check_release_readiness()

    su.log_info("Release Readiness Check")
    su.log_info("=" * 50)
    su.log_info(f"  Ready for release: {is_ready}")

    if issues:
        su.log_info(f"  Issues found ({len(issues)}):")
        for issue in issues:
            su.log_info(f"    - {issue}")
    else:
        su.log_info("  No issues found -- package is release-ready")
finally:
    os.chdir(original_cwd)

In [ ]:
# --- 7c. Version increment simulation (no files modified) ---
from siege_utilities.hygiene.pypi_release import increment_version

if current_version:
    su.log_info("Version Increment Simulation")
    su.log_info("=" * 50)
    su.log_info(f"  Current version:   {current_version}")
    su.log_info(f"  Patch increment:   {increment_version(current_version, 'patch')}")
    su.log_info(f"  Minor increment:   {increment_version(current_version, 'minor')}")
    su.log_info(f"  Major increment:   {increment_version(current_version, 'major')}")
else:
    su.log_info("Skipping version increment simulation -- no version found")

## 8. Summary

### Modules & Key Functions

| Module | Function | Purpose |
|--------|----------|---------|
| `development/architecture` | `analyze_package_structure()` | Introspect package modules, functions, classes |
| `development/architecture` | `generate_architecture_diagram()` | Produce text/JSON/Markdown/RST diagrams |
| `git/branch_analyzer` | `analyze_branch_status()` | Current branch, ahead/behind, last commit |
| `git/branch_analyzer` | `get_commit_history()` | Recent commits with hash, date, author, message |
| `git/branch_analyzer` | `get_file_changes()` | Added/modified/deleted files vs main |
| `git/branch_analyzer` | `generate_branch_report()` | Full Markdown branch status report |
| `git/git_status` | `get_repository_status()` | Comprehensive repo state snapshot |
| `git/git_status` | `get_branch_info()` | All local and remote branches |
| `git/git_status` | `get_remote_info()` | Remote URLs and default remote |
| `git/git_status` | `get_file_status()` | Staged, unstaged, untracked file lists |
| `git/git_status` | `get_repository_size()` | .git and working tree sizes in MB |
| `git/git_workflow` | `validate_branch_naming()` | Check branch names against convention patterns |
| `git/git_workflow` | `enforce_commit_conventions()` | Validate commit messages (conventional commits) |
| `testing/environment` | `get_system_info()` | Python, Java, env vars, platform details |
| `testing/environment` | `diagnose_environment()` | Full environment health check |
| `testing/runner` | `quick_smoke_test()` | Basic package functionality verification |
| `testing/runner` | `get_test_report()` | Comprehensive test and environment report |
| `hygiene/generate_docstrings` | `analyze_function_signature()` | Extract parameter types and return info |
| `hygiene/generate_docstrings` | `generate_docstring_template()` | Produce standardized docstring from signature |
| `hygiene/pypi_release` | `get_current_version()` | Read version from setup.py |
| `hygiene/pypi_release` | `check_release_readiness()` | Validate package is ready for release |

### Safety Notes

Functions intentionally **excluded** from this notebook (they perform write operations):

| Module | Function | Reason Excluded |
|--------|----------|-----------------|
| `git/git_workflow` | `start_feature_workflow()` | Creates branches, pushes to remote |
| `git/git_workflow` | `complete_feature_workflow()` | Merges and deletes branches |
| `git/git_workflow` | `hotfix_workflow()` | Creates branches, pushes to remote |
| `git/git_workflow` | `release_workflow()` | Creates branches, updates version files |
| `hygiene/pypi_release` | `full_release_workflow()` | Modifies files, builds, uploads |
| `hygiene/pypi_release` | `upload_to_pypi()` | Uploads to package registry |
| `hygiene/generate_docstrings` | `process_python_file()` | Modifies source files |

### Pattern: Read-Only Exploration

```python
# Always point git functions at a known repo
REPO_PATH = str(Path.cwd().parent)

# Read-only analysis
status = get_repository_status(repo_path=REPO_PATH)
commits = get_commit_history(limit=10, repo_path=REPO_PATH)

# Validation on sample data (no git ops)
result = validate_branch_naming("feature/my-feature")
result = enforce_commit_conventions("feat: add new parser")

# Release checks without modification
version = get_current_version("setup.py")
ready, issues = check_release_readiness()
```